# Khmer Spell Checker: training on Kaggle (GPU T4 x2)

Runs the whole pipeline (`python main.py`): data, lexicon, G2P, character tagger, ByT5, ranking + evaluation.

## Before you run (once)
1. **Settings → Accelerator: GPU T4 x2**, **Internet: on**.
2. **Hugging Face login**, one of:
   - **Interactive run** (you click *Run* yourself): the login cell shows a box, paste a token with **write** permission.
     Run that cell, log in, then run the rest.
   - **Background run** (*Save Version → Save & Run All*): nobody can type into the box, so add the token as
     **Add-ons → Secrets → `HF_TOKEN`** and tick it for this notebook. The login cell uses it automatically.
3. **Add-ons → Secrets → `GITHUB_TOKEN`**: only if your GitHub repo is **private**.

## Checkpoints and resume (Kaggle: 12 h per session, ~30 GPU hours per week)
Everything is saved to the **public** model repo `MODEL_REPO` on Hugging Face:

| In the repo | What | When |
|---|---|---|
| `last-checkpoint/` | newest ByT5 checkpoint (model + optimizer) | every 2,000 steps, by the trainer |
| root | ByT5 model | every 2,000 steps, final at the end |
| `speller/` | final tagger, lexicon, G2P, ranking weights | at the end |
| `training-state/` | data, lexicon, G2P/tagger resume states (`*.resume`) | every 10 min, by this notebook |

**After any stop** (12 h limit, network error, crash): just **run the notebook again**
(best: *Save Version → Save & Run All*, it runs in the background). It downloads `training-state/`,
`main.py` skips finished steps, the G2P and the tagger continue from their `.resume` state
(the tagger even in the middle of an epoch) and ByT5 continues from `last-checkpoint/`.

**After a code change of the data** (e.g. names-v1): models trained for the old data or dictionary
(`<model>.version` in `training-state/`) are trained again from scratch automatically.

The log prints the real speed (`it/s`) of ByT5 training: use it to choose `BYT5_STEPS`.

In [ ]:
# ---------------- settings ----------------
GITHUB_REPO = "https://github.com/PhonSobon/Khmer-Spelling-Check.git"   # <- your repository
BRANCH = "main"

QUICK = False            # True: 10% data + short training, to test everything first (~1-2 h)
BYT5_STEPS = 20000       # ByT5 training steps (3090 default: 30000). Continues across sessions.
PER_DEVICE_BATCH = 16    # ByT5 batch per T4 (16 GB, fp32). Total batch stays 64. Lower to 8 if "CUDA out of memory".
PUSH_MODELS = True       # push checkpoints + models to MODEL_REPO (needed for resume of ByT5 across sessions)
MODEL_REPO = "phonsobon/khmer-spell-checker"  # PUBLIC HF model repo: models, checkpoints, training state
SYNC_MINUTES = 10        # how often outputs/ is copied to MODEL_REPO/training-state/

In [ ]:
# ---------------- Hugging Face login ----------------
# Interactive: a login box appears -> paste a WRITE token, then run the next cells.
# Background run (Save & Run All): uses the Kaggle secret HF_TOKEN instead (no box possible there).
import os
from huggingface_hub import get_token, notebook_login


def _secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None


_hf_secret = _secret("HF_TOKEN")
if _hf_secret:
    os.environ["HF_TOKEN"] = _hf_secret
    print("logged in with the Kaggle secret HF_TOKEN")
elif get_token():
    print("already logged in to Hugging Face")
else:
    notebook_login()      # does not wait: log in in the box, THEN run the next cell

In [ ]:
# ---------------- check the login (run after logging in above) ----------------
from huggingface_hub import HfApi

_token = os.environ.get("HF_TOKEN") or get_token()
if not _token:
    raise SystemExit("Not logged in yet: log in with the box above, then run this cell again.")
os.environ["HF_TOKEN"] = _token        # main.py and its training scripts read the token from here
print("Hugging Face user:", HfApi(token=_token).whoami()["name"])
_GH = _secret("GITHUB_TOKEN")          # only needed for a private GitHub repo
print("GITHUB_TOKEN loaded" if _GH else "no GITHUB_TOKEN (public GitHub repo)")

In [ ]:
# ---------------- code: clone into /tmp (outside the saved output folder) ----------------
import subprocess, sys
from pathlib import Path

CODE = Path("/tmp/Khmer-Spelling-Check")
if (CODE / ".git").exists():          # same session, run again: take the newest code (also after a force-push)
    subprocess.run(["git", "-C", str(CODE), "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(CODE), "reset", "--hard", "FETCH_HEAD"], check=True)
else:
    url = GITHUB_REPO.replace("https://", f"https://x-access-token:{_GH}@") if _GH else GITHUB_REPO
    r = subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, str(CODE)], capture_output=True, text=True)
    if r.returncode:
        raise SystemExit(r.stderr.replace(_GH or "\0", "***"))     # never print the token
    subprocess.run(["git", "-C", str(CODE), "remote", "set-url", "origin", GITHUB_REPO], check=True)
print(subprocess.run(["git", "-C", str(CODE), "log", "-1", "--format=%h %s (%cr)"], capture_output=True, text=True).stdout)

In [ ]:
# ---------------- packages (Kaggle already has torch + CUDA: do not reinstall torch) ----------------
!pip install -q khmer-nltk python-dotenv hf_xet "transformers>=4.46" "datasets>=2.19"

In [ ]:
# ---------------- GPU check ----------------
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
!cd /tmp/Khmer-Spelling-Check && python scripts/check_gpu.py

In [ ]:
# ---------------- restore outputs/ from the previous session ----------------
import time
from huggingface_hub import HfApi, snapshot_download

STATE = "training-state"                       # folder in MODEL_REPO
OUT = Path("/kaggle/working") / STATE           # = outputs/ of main.py
OUT.mkdir(parents=True, exist_ok=True)
os.environ["KHSPELL_OUTPUTS"] = str(OUT)

api = HfApi(token=os.environ["HF_TOKEN"])
print("logged in as", api.whoami()["name"])
api.create_repo(MODEL_REPO, private=False, exist_ok=True)   # public


def retry(fn, what, attempts=6):
    for i in range(1, attempts + 1):
        try:
            return fn()
        except Exception as e:
            if i == attempts:
                raise
            print(f"{what} failed ({type(e).__name__}), retry {i}/{attempts - 1} in {20 * i}s", flush=True)
            time.sleep(20 * i)


files = retry(lambda: api.list_repo_files(MODEL_REPO), "listing the repo")
if any(f.startswith(STATE + "/") for f in files):
    print(f"restoring {STATE}/ from {MODEL_REPO} ...")
    retry(lambda: snapshot_download(MODEL_REPO, allow_patterns=f"{STATE}/**", local_dir="/kaggle/working",
                                    token=os.environ["HF_TOKEN"]), "download")
    for p in sorted(OUT.iterdir()):
        print("  ", p.name)
else:
    print(f"first session: nothing to restore in {MODEL_REPO}/{STATE}/")
if "last-checkpoint/trainer_state.json" in files:
    print("ByT5 checkpoint found in the repo: main.py will continue from it")

In [ ]:
# ---------------- background sync: outputs/ -> MODEL_REPO/training-state/ every SYNC_MINUTES ----------------
import threading

TRAINER_PUSHES = PUSH_MODELS and not QUICK      # then the trainer itself uploads ByT5 checkpoints (last-checkpoint/)


def _step(path):
    return int(path.name.rsplit("-", 1)[-1])


def sync(note="sync"):
    now = time.time()
    ignore = [".cache/**", "**/.ipynb_checkpoints/**", "**/*.tmp", "**/_hub_download/**"]
    # files still being written: skip this round
    ignore += [f.relative_to(OUT).as_posix() for f in OUT.rglob("*") if f.is_file() and now - f.stat().st_mtime < 60]
    delete = []
    for model_dir in OUT.glob("topk_model*"):
        ckpts = sorted(model_dir.glob("checkpoint-*"), key=_step)
        ready = [c for c in ckpts if (c / "trainer_state.json").exists()
                 and now - (c / "trainer_state.json").stat().st_mtime > 60]
        keep = None if TRAINER_PUSHES else (ready[-1] if ready else None)   # else: only the newest one
        ignore += [f"{c.relative_to(OUT).as_posix()}/**" for c in ckpts if c != keep]
        if keep:
            delete.append(f"{model_dir.name}/checkpoint-*/**")     # older checkpoints in the repo (relative to STATE/)
    # files that are gone here (a finished .resume state, a model trained again from scratch) are deleted in
    # the repo too, or the next session would restore them and redo or skip a step by mistake
    prefix = STATE + "/"
    delete += [f[len(prefix):] for f in api.list_repo_files(MODEL_REPO)
               if f.startswith(prefix) and "/checkpoint-" not in f and not (OUT / f[len(prefix):]).exists()]
    api.upload_folder(repo_id=MODEL_REPO, folder_path=str(OUT), path_in_repo=STATE, ignore_patterns=ignore,
                      delete_patterns=delete or None, commit_message=note)
    print(f"[{time.strftime('%H:%M')}] {STATE}/ synced to {MODEL_REPO}", flush=True)


_stop = threading.Event()


def _loop():
    while not _stop.wait(SYNC_MINUTES * 60):
        try:
            sync()
        except Exception as e:                       # a failed sync must never stop training
            print(f"sync failed (will retry): {e}", flush=True)


threading.Thread(target=_loop, daemon=True).start()
print(f"syncing {STATE}/ to https://huggingface.co/{MODEL_REPO} every {SYNC_MINUTES} min")

In [ ]:
# ---------------- train: python main.py (finished steps are skipped) ----------------
cmd = [sys.executable, "-u", "main.py", "--per-device-batch", str(PER_DEVICE_BATCH)]
if QUICK:
    cmd.append("--quick")
else:
    cmd += ["--steps", str(BYT5_STEPS)]
if TRAINER_PUSHES:
    cmd += ["--push-to-hub", MODEL_REPO]      # checkpoints + models to MODEL_REPO, resume from last-checkpoint/
elif not PUSH_MODELS:
    cmd.append("--no-push")
print(" ".join(cmd[1:]), flush=True)

env = dict(os.environ, PYTHONIOENCODING="utf-8", TOKENIZERS_PARALLELISM="false")
proc = subprocess.Popen(cmd, cwd=CODE, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, encoding="utf-8", errors="replace")
for line in proc.stdout:
    print(line, end="", flush=True)
code_ = proc.wait()
_stop.set()
sync("end of session")
print("\nFINISHED" if code_ == 0 else f"\nmain.py stopped with exit code {code_}: see the log above, fix, run again")

In [ ]:
# ---------------- results (end of outputs/main.log) ----------------
log = (OUT / "main.log").read_text(encoding="utf-8", errors="replace").splitlines()
start = max((i for i, l in enumerate(log) if "7/7 Tune ranking" in l), default=max(0, len(log) - 80))
print("\n".join(log[start:]))

In [ ]:
# ---------------- try the spell checker ----------------
os.environ["KHSPELL_TOPK_MODEL"] = str(OUT / ("topk_model_quick" if QUICK else "topk_model"))
os.environ["KHSPELL_TAGGER"] = str(OUT / ("tagger_quick.pt" if QUICK else "tagger.pt"))
sys.path.insert(0, str(CODE))
os.chdir(CODE)
from khspell.topk import Speller

sp = Speller.load()
for text in ["ខ្ញុំជាសស្សដែលមានសក្សានៅកម្ពុច",
             "ប្រជាពលរដ្ធខ្មែរទាំងអស់",
             "លោករដ្ឋលេខាតិកាបានចូលរួមប្រជុំ",
             "ការអប់រំមានសារះសំខាន់សម្រាប់សង្គម។"]:
    _, errors = sp.check(text, k=3)
    print(text)
    for e in errors:
        print(f"   {e['word']:14s} -> {[w for w, _ in e['suggestions']]}")